In [ ]:
!pip install xgboost
!pip install lightgbm
!pip install catboost

In [ ]:
# LUNG Cancer with ONLY researched biomarkers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Load
df = pd.read_csv("lung_cancer_extended.csv")

# ── Core Lung biomarkers only ──────────────────────────────
core_features = [
    "Age",                   # Age
    "Smoking",               # Tobacco
    "Air_Pollution",         # Radon proxy
    "Occupational_Hazards",  # Second-Hand Smoke proxy
    "Family_History",        # Family History
]

target_col = "Risk_Level"

X  = df[core_features]
le = LabelEncoder().fit(df[target_col])
y  = le.transform(df[target_col])

print(f"Features: {core_features}")
print(f"Shape: {X.shape}")
print(df[target_col].value_counts())

# Split + Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, max_depth=6,
                                                   min_samples_leaf=10,
                                                   class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42,
                                                   max_depth=10, min_samples_leaf=5,
                                                   class_weight='balanced'),
    "Extra Trees":         ExtraTreesClassifier(n_estimators=200, random_state=42,
                                                max_depth=10, min_samples_leaf=5,
                                                class_weight='balanced'),
    "AdaBoost":            AdaBoostClassifier(n_estimators=150, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=150, random_state=42,
                                                       max_depth=4, learning_rate=0.05,
                                                       subsample=0.8),
    "SVM (RBF Kernel)":    SVC(kernel='rbf', probability=True, random_state=42,
                               class_weight='balanced'),
    "XGBoost":             XGBClassifier(n_estimators=150, random_state=42,
                                          max_depth=4, learning_rate=0.05,
                                          subsample=0.8, reg_alpha=0.1,
                                          reg_lambda=1.0, eval_metric='logloss'),
    "LightGBM":            LGBMClassifier(n_estimators=150, random_state=42,
                                           max_depth=4, learning_rate=0.05,
                                           reg_alpha=0.5, reg_lambda=2.0,
                                           min_child_samples=30,
                                           class_weight='balanced'),
    "CatBoost":            CatBoostClassifier(verbose=0, random_state=42,
                                               depth=4, learning_rate=0.05,
                                               l2_leaf_reg=5.0)
}

# Train and evaluate
results     = {}
predictions = {}

print("\n=== Results: Core Lung Features Only ===")
print(f"{'Model':<25} {'Test Acc':>10} {'Train Acc':>10} {'Gap':>8} {'Status':>10}")
print("-" * 68)

for name, model in models.items():
    if name == "XGBoost":
        sw = compute_sample_weight(class_weight='balanced', y=y_train)
        model.fit(X_train, y_train, sample_weight=sw)
    else:
        model.fit(X_train, y_train)

    y_pred    = model.predict(X_test)
    test_acc  = accuracy_score(y_test, y_pred)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    gap       = train_acc - test_acc
    status    = "⚠️ Overfit" if gap > 0.15 else "⚡ Slight" if gap > 0.05 else "✓ Good"

    results[name]     = test_acc
    predictions[name] = y_pred
    print(f"{name:<25} {test_acc:>10.4f} {train_acc:>10.4f} {gap:>8.4f} {status:>10}")

# Best model report
results_df      = pd.DataFrame(list(results.items()),
                                columns=["Model", "Accuracy"]).sort_values(
                                by="Accuracy", ascending=False)
best_model_name = results_df.iloc[0]['Model']

print(f"\n🏁 Best Model: {best_model_name} — {results_df.iloc[0]['Accuracy']:.4f}")
print(f"\n=== Classification Report: {best_model_name} ===")
print(classification_report(y_test, predictions[best_model_name],
                             target_names=le.classes_))

Features: ['Age', 'Smoking', 'Air_Pollution', 'Occupational_Hazards', 'Family_History']
Shape: (4471, 5)
Risk_Level
Medium    3721
High       459
Low        291
Name: count, dtype: int64

=== Results: Core Lung Features Only ===
Model                       Test Acc  Train Acc      Gap     Status
--------------------------------------------------------------------
Logistic Regression           0.1932     0.2153   0.0221     ✓ Good
Decision Tree                 0.2710     0.3051   0.0341     ✓ Good
Random Forest                 0.6530     0.7897   0.1368   ⚡ Slight
Extra Trees                   0.4454     0.5559   0.1105   ⚡ Slight
AdaBoost                      0.8327     0.8321  -0.0006     ✓ Good
Gradient Boosting             0.8309     0.8410   0.0101     ✓ Good
SVM (RBF Kernel)              0.2397     0.2693   0.0296     ✓ Good
XGBoost                       0.4240     0.5231   0.0991   ⚡ Slight
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.00

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM                      0.4070     0.4834   0.0765   ⚡ Slight
CatBoost                      0.8336     0.8357   0.0020     ✓ Good

🏁 Best Model: CatBoost — 0.8336

=== Classification Report: CatBoost ===
              precision    recall  f1-score   support

        High       1.00      0.01      0.02       110
         Low       0.00      0.00      0.00        77
      Medium       0.83      1.00      0.91       931

    accuracy                           0.83      1118
   macro avg       0.61      0.34      0.31      1118
weighted avg       0.79      0.83      0.76      1118



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
